# GymRAVANA pose dataset preparation

This notebook audits the supplied 18-image starter and extracts MediaPipe landmarks for a separate **pose identity** experiment. It does not create progression-readiness labels and it does not treat reference-collage labels as trainer-verified form assessments.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'artisan').exists())
sys.path.insert(0, str(PROJECT_ROOT))
from ai.pose.workflow import EXCLUDED_POSES, POSE_ALIASES, prepare_features

STARTER_DIR = PROJECT_ROOT / 'ai/data/pose_starter'
MODEL_PATH = PROJECT_ROOT / 'ai/models/pose_landmarker_lite.task'
OUTPUT_CSV = PROJECT_ROOT / 'ai/data/pose_features.csv'
OUTPUT_METADATA = PROJECT_ROOT / 'ai/data/pose_features.metadata.json'

In [2]:
annotations = pd.read_csv(STARTER_DIR / 'pose_annotations.csv')
print('Raw rows:', len(annotations))
print('Source groups:', annotations['source_id'].nunique())
print('Form scores present:', int(annotations['form_score'].notna().sum()))
print('Trainer-verified rows:', int(annotations['score_verified_by_trainer'].eq('yes').sum()))
print('Rows allowed for final claims:', int(annotations['usable_for_final_academic_claims'].eq('yes').sum()))
print('Class/source table:')
display(pd.crosstab(annotations['pose_name'], annotations['source_id']))
print('Excluded because its references conflict:', sorted(EXCLUDED_POSES))
print('Canonical class mapping:', json.dumps(POSE_ALIASES, indent=2))

Raw rows: 18
Source groups: 3
Form scores present: 0
Trainer-verified rows: 0
Rows allowed for final claims: 0
Class/source table:


source_id,source_01_incorrect_examples,source_02_correct_examples,source_03_correct_examples
pose_name,,,
adho_mukha_virasana,1,1,1
chakrasana,1,1,1
mayurasana,1,1,1
santulanasana,1,1,1
shirshasana,1,1,1
virasana,1,1,1


Excluded because its references conflict: ['santulanasana']
Canonical class mapping: {
  "virasana": "virasana",
  "adho_mukha_virasana": "balasana",
  "chakrasana": "urdhva_dhanurasana",
  "mayurasana": "mayurasana",
  "shirshasana": "salamba_sirsasana"
}


In [3]:
preparation = prepare_features(STARTER_DIR, MODEL_PATH, OUTPUT_CSV, OUTPUT_METADATA)
print(json.dumps(preparation, indent=2))
assert preparation['row_count'] >= 10, 'Too few poses were detected for even a starter experiment.'
assert preparation['class_count'] == 5
assert preparation['source_group_count'] == 3

{
  "schema_version": 1,
  "task": "five_class_yoga_pose_identity",
  "target": "canonical_pose",
  "row_count": 15,
  "class_count": 5,
  "source_group_count": 3,
  "classes": [
    "balasana",
    "mayurasana",
    "salamba_sirsasana",
    "urdhva_dhanurasana",
    "virasana"
  ],
  "columns": [
    "sample_id",
    "source_id",
    "source_kind",
    "pose_name",
    "canonical_pose",
    "form_class",
    "left_elbow_angle",
    "right_elbow_angle",
    "left_shoulder_angle",
    "right_shoulder_angle",
    "left_hip_angle",
    "right_hip_angle",
    "left_knee_angle",
    "right_knee_angle",
    "torso_angle_deg",
    "shoulder_slope_deg",
    "hip_slope_deg",
    "elbow_symmetry_abs",
    "shoulder_symmetry_abs",
    "hip_symmetry_abs",
    "knee_symmetry_abs",
    "shoulder_width_norm",
    "hip_width_norm",
    "wrist_distance_norm",
    "ankle_distance_norm",
    "visibility_mean"
  ],
  "feature_names": [
    "left_elbow_angle",
    "right_elbow_angle",
    "left_shoulder_an

## Interpretation

The resulting table contains derived joint geometry rather than names, raw pixels or medical information. Its integrity metadata binds the CSV to the source annotations and the exact MediaPipe model bundle. Fifteen possible rows are still far below a defensible deployment dataset, so this output is for pipeline prototyping only.